<a href="https://colab.research.google.com/github/SlanderDome/Foliage_Care/blob/master/newcolab_train_disease_model_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import os
from google.colab import files

# --- 1. Setup ---
# Unzip your 'new_potato_disease_dataset.zip' file first.
# Make sure your FILE_ID is set correctly.

# Example of how to unzip (run this in a separate cell):
!pip install gdown
FILE_ID = "1pYNi5gtr5xlKC9xJ7Oj2cU-BHkz14zqG"
!gdown --id {FILE_ID} -O "new_potato_disease_dataset.zip"
!unzip -q "new_potato_disease_dataset.zip" -d "/content/"

# --- THIS IS THE UPDATE ---
# This script assumes your data is in a folder named 'new_potato_disease_dataset'
train_dir = "/content/new_potato_disease_dataset"
# --- END UPDATE ---

if not os.path.exists(train_dir):
    print(f"Error: Directory not found at '{train_dir}'")
    print("Please make sure you have unzipped your dataset and the path is correct.")
else:
    print("Dataset directory found.")

    # --- 2. Data Augmentation & Generators ---
    # We will use powerful augmentation to make the model robust
    train_datagen_augmented = ImageDataGenerator(
        rescale=1.0/255,
        validation_split=0.2,    # Use 20% of data for validation
        rotation_range=40,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    # Validation generator (no augmentation, just rescaling)
    validation_datagen = ImageDataGenerator(
        rescale=1.0/255,
        validation_split=0.2
    )

    print("Loading Training Data...")
    train_data = train_datagen_augmented.flow_from_directory(
        train_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical', # Switched to categorical for multi-class
        subset='training'
    )

    print("Loading Validation Data...")
    val_data = validation_datagen.flow_from_directory(
        train_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
        subset='validation'
    )

    # Get the number of classes
    num_classes = len(train_data.class_indices)
    print(f"\nFound {num_classes} classes.")
    print("Class indices:", train_data.class_indices)


    # --- 3. Build the Model (with Fine-Tuning) ---

    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

    # Unfreeze the top layers for fine-tuning
    base_model.trainable = True
    fine_tune_at = 100
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False

    # Build model
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax') # Use softmax for multi-class
    ])

    # Compile with a smaller learning rate for fine-tuning
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    # --- 4. Callbacks for Better Training ---

    # Stop training if validation accuracy doesn't improve
    early_stopper = EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        verbose=1,
        mode='max',
        restore_best_weights=True
    )

    # Reduce learning rate if training plateaus
    lr_reducer = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        verbose=1,
        min_lr=1e-6
    )

    # --- 5. Train the Model ---
    print("\nStarting model training...")

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=30, # Set a high number, EarlyStopping will find the best
        callbacks=[early_stopper, lr_reducer]
    )

    # --- 6. Save and Download ---
    model_filename = "new_disease_model.h5"
    model.save(model_filename)
    print(f"✅ Model saved as {model_filename}")

    print(f"Downloading {model_filename}...")
    files.download(model_filename)

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1pYNi5gtr5xlKC9xJ7Oj2cU-BHkz14zqG
From (redirected): https://drive.google.com/uc?id=1pYNi5gtr5xlKC9xJ7Oj2cU-BHkz14zqG&confirm=t&uuid=0b50ce19-de8e-41bd-9989-3d898a758974
To: /content/new_potato_disease_dataset.zip
100% 241M/241M [00:01<00:00, 180MB/s]
Dataset directory found.
Loading Training Data...
Found 11629 images belonging to 22 classes.
Loading Validation Data...
Found 2896 images belonging to 22 classes.

Found 22 classes.
Class indices: {'Apple___Apple_scab': 0, 'Apple___Black_rot': 1, 'Apple___Cedar_apple_rust': 2, 'Apple___healthy': 3, 'Cherry_(including_sour)___Powdery_mildew': 4, 'Cherry_(including_sour)___healthy': 5, 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 6, 'Corn_(maize)___Com

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 22)             │         2,838 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,424,790 (9.25 MB)

 Trainable params: 2,028,246 (7.74 MB)

...

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
364/364 ━━━━━━━━━━━━━━━━━━━━ 201s 468ms/step - accuracy: 0.5403 - loss: 1.6093 - val_accuracy: 0.5653 - val_loss: 1.5159 - learning_rate: 1.0000e-04
Epoch 2/30
364/364 ━━━━━━━━━━━━━━━━━━━━ 143s 393ms/step - accuracy: 0.9365 - loss: 0.2260 - val_accuracy: 0.7880 - val_loss: 0.7526 - learning_rate: 1.0000e-04
Epoch 3/30
364/364 ━━━━━━━━━━━━━━━━━━━━ 140s 384ms/step - accuracy: 0.9533 - loss: 0.1513 - val_accuracy: 0.9175 - val_loss: 0.2784 - learning_rate: 1.0000e-04
Epoch 4/30
364/364 ━━━━━━━━━━━━━━━━━━━━ 139s 383ms/step - accuracy: 0.9703 - loss: 0.0913 - val_accuracy: 0.9648 - val_loss: 0.1053 - learning_rate: 1.0000e-04
Epoch 5/30
364/364 ━━━━━━━━━━━━━━━━━━━━ 142s 389ms/step - accuracy: 0.9747 - loss: 0.0836 - val_accuracy: 0.9682 - val_loss: 0.1084 - learning_rate: 1.0000e-04
Epoch 6/30
364/364 ━━━━━━━━━━━━━━━━━━━━ 141s 388ms/step - accuracy: 0.9791 - loss: 0.0704 - val_accuracy: 0.9738 - val_loss: 0.0920 - learning_rate: 1.0000e-04
Epoch 7/30
364/364 ━━━━━━━━━━━━━━━━━━━━ 

✅ Model saved as new_disease_model.h5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>